In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.


In [2]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/fedesoriano/cifar100"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/

/home/ec2-user/SageMaker


In [41]:
# Import pip packages
import pandas as pd
import numpy as np
import torch
import os
import pickle
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader
from PIL import Image

In [4]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [5]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [11]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

In [17]:
data = unpickle('./dataset/train')
print(data.keys())

dict_keys([b'filenames', b'batch_label', b'fine_labels', b'coarse_labels', b'data'])


In [21]:
metadata_path = './dataset/meta'
metadata = unpickle(metadata_path)
CLASSES = dict(list(enumerate(metadata[b'fine_label_names'])))
CLASSES

{0: b'apple',
 1: b'aquarium_fish',
 2: b'baby',
 3: b'bear',
 4: b'beaver',
 5: b'bed',
 6: b'bee',
 7: b'beetle',
 8: b'bicycle',
 9: b'bottle',
 10: b'bowl',
 11: b'boy',
 12: b'bridge',
 13: b'bus',
 14: b'butterfly',
 15: b'camel',
 16: b'can',
 17: b'castle',
 18: b'caterpillar',
 19: b'cattle',
 20: b'chair',
 21: b'chimpanzee',
 22: b'clock',
 23: b'cloud',
 24: b'cockroach',
 25: b'couch',
 26: b'crab',
 27: b'crocodile',
 28: b'cup',
 29: b'dinosaur',
 30: b'dolphin',
 31: b'elephant',
 32: b'flatfish',
 33: b'forest',
 34: b'fox',
 35: b'girl',
 36: b'hamster',
 37: b'house',
 38: b'kangaroo',
 39: b'keyboard',
 40: b'lamp',
 41: b'lawn_mower',
 42: b'leopard',
 43: b'lion',
 44: b'lizard',
 45: b'lobster',
 46: b'man',
 47: b'maple_tree',
 48: b'motorcycle',
 49: b'mountain',
 50: b'mouse',
 51: b'mushroom',
 52: b'oak_tree',
 53: b'orange',
 54: b'orchid',
 55: b'otter',
 56: b'palm_tree',
 57: b'pear',
 58: b'pickup_truck',
 59: b'pine_tree',
 60: b'plain',
 61: b'plate',

In [47]:
class CustomDataset(Dataset):
    def __init__(self, data, transform=None):
        self.images = data[b'data']
        self.labels = data[b'fine_labels']
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if len(image.shape) == 1:
            image = image.reshape(3, 32, 32).transpose(1, 2, 0)

        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)

In [ ]:
train = unpickle('./dataset/train')
train_dataset = CustomDataset(train, train_transform)
_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)

In [48]:
# Find mean, std
mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

mean, std

(tensor([-0.1309, -0.1705, -0.2517]), tensor([0.5924, 0.5669, 0.5778]))

In [72]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 50,
        'batch_size': 64,
        'lr': 0.001,
        'workers': 2,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-03-10-40-01-158


2026-07-03 10:40:05 Starting - Starting the training job...
2026-07-03 10:40:19 Starting - Preparing the instances for training...
2026-07-03 10:40:46 Downloading - Downloading input data...
2026-07-03 10:41:16 Downloading - Downloading the training image.........
2026-07-03 10:43:07 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 560.35.05
Current installed NVIDIA driver version is 570.211.01
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-07-03 10:43:13,579 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-07-03 10:43:13,600 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-07-03 10:43:13,610 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succe

In [73]:
# Create new model + Endpoint Configuration
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
  name = f'cifar100-model-{int(time.time())}',
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'cifar100-endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-200148130345/sagemaker/output/pytorch-training-2026-07-03-10-40-01-158/output/model.tar.gz), script artifact (s3://sagemaker-us-east-1-200148130345/pytorch-training-2026-07-03-10-40-01-158/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-200148130345/cifar100-model-1783076266/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: cifar100-model-1783076266


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/cifar100-endpoint-configuration-1783076268',
 'ResponseMetadata': {'RequestId': 'a20fcede-d355-4945-8ad3-ecdfd8795609',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a20fcede-d355-4945-8ad3-ecdfd8795609',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '123',
   'date': 'Fri, 03 Jul 2026 10:57:49 GMT'},
  'RetryAttempts': 0}}

In [74]:
# Create/Update Endpoint
# sm.create_endpoint(
#     EndpointName = "cifar100-endpoint",
#     EndpointConfigName = new_config
# )

sm.update_endpoint(
    EndpointName = "cifar100-endpoint",
    EndpointConfigName = new_config
)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/cifar100-endpoint',
 'ResponseMetadata': {'RequestId': 'e120939e-5752-498b-b02b-393a44cb4b18',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e120939e-5752-498b-b02b-393a44cb4b18',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '85',
   'date': 'Fri, 03 Jul 2026 10:57:52 GMT'},
  'RetryAttempts': 0}}

In [79]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar100-endpoint'
image_path = "./assets/apple.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 0, 'confidence': 0.9105933308601379}
b'apple'


In [80]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar100-endpoint'
image_path = "./assets/dinosaur.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 29, 'confidence': 0.9999972581863403}
b'dinosaur'


In [81]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar100-endpoint'
image_path = "./assets/elephant.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 31, 'confidence': 0.8205728530883789}
b'elephant'


In [82]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar100-endpoint'
image_path = "./assets/sea.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 71, 'confidence': 0.9110639095306396}
b'sea'


In [83]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar100-endpoint'
image_path = "./assets/train.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 90, 'confidence': 0.7237533330917358}
b'train'
